# Week 10 · Influence, Communities & Information Diffusion
**ISB02306504 – Social Media Mining** · Information Systems for Business · Odd Semester 2026/2027

This week asks three stakeholder questions:

1. **Influence:** *who matters, and in what sense?* (centrality)
2. **Communities:** *is the conversation one space or several groups?* (components, Louvain, modularity)
3. **Diffusion:** *how did a message actually spread?* (cascades: reach, depth, breadth, speed)

| Part | Data | Slides |
|---|---|---|
| 1 | The slide toy graph: reproduce every number on the slides | 3–33 |
| 2 | Zachary's karate club (built into NetworkX): do detected communities match a real split? | 22–33 |
| 3 | **Higgs Twitter dataset** (SNAP): 563k retweets, mentions and replies around the Higgs boson announcement, July 2012. Influence, communities, cascades, and centrality vs observed diffusion | 3–45 |
| 4 | Stakeholder summary, class activity, homework | 41–45 |

The Higgs data is read **directly from the SNAP website with pandas**.

---
## 0. Setup

In [ ]:
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import spearmanr
from sklearn.metrics import normalized_mutual_info_score
from IPython.display import display

SEED = 42
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 11, "font.size": 9})
BLUE, ORANGE, GREY, GREEN, PINK = "#2a78d6", "#eb6834", "#8a8984", "#1baf7a", "#e87ba4"
COMM_COLORS = [BLUE, ORANGE, GREEN, "#eda100", PINK, "#6250d6", "#008300", "#e34948", "#5b3a29", "#17becf"]
print("networkx", nx.__version__, "| pandas", pd.__version__)

---
## 1. The slide graph: four lenses on "important"
**Influence is not a single graph property.** Start from the behaviour the stakeholder cares about (slide 3):

| Role | Question | Network clue |
|---|---|---|
| Visible | receives many mentions / replies / interactions? | degree, in-degree |
| Broker | connects otherwise separated groups? | betweenness |
| Reachable | can reach many others in few steps? | closeness |
| Prestigious | connected to other important accounts? | eigenvector, PageRank |

In [ ]:
G = nx.Graph()
G.add_edges_from([("A", "B"), ("A", "C"), ("B", "C"), ("C", "D"),
                  ("D", "E"), ("D", "F"), ("E", "F"), ("D", "G"),
                  ("G", "H"), ("G", "I"), ("H", "I")])
POS = {"A": (0, 1), "B": (0, -1), "C": (1, 0), "D": (2, 0), "E": (1.6, 1.2), "F": (2.4, 1.2),
       "G": (3, 0), "H": (4, 1), "I": (4, -1)}

lenses = pd.DataFrame({
    "degree": nx.degree_centrality(G),
    "betweenness": nx.betweenness_centrality(G),
    "closeness": nx.closeness_centrality(G),
    "eigenvector": nx.eigenvector_centrality(G, max_iter=1000),
}).round(3)
display(lenses.sort_values("betweenness", ascending=False))

Compare with slide 18: D = 0.500 / 0.750 / 0.667 / 0.535. Check one value by hand. **Normalised degree** of D = 4 neighbours / (n − 1) = 4 / 8 = 0.50 (slide 6). Normalising makes a degree of 20 in a 30-node graph comparable with one in a 30,000-node graph.

Now draw the graph once per lens, with node size = score:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, col in zip(axes, lenses.columns):
    s = lenses[col]
    nx.draw_networkx(G, POS, ax=ax, node_size=[200 + 2200 * v for v in s], node_color=BLUE,
                     font_color="white", edge_color=GREY)
    ax.set_title(f"{col}  (top: {s.idxmax()} = {s.max():.2f})"); ax.axis("off")
plt.tight_layout(); plt.show()

In this toy graph D wins on every lens. **In real networks the rankings usually disagree**, and the disagreement is the interesting part (we will see it in part 3). Notice already: E/F have the same degree as A/B (0.25) but a higher closeness (0.44 vs 0.38) because they sit next to D, and C and G carry bridge traffic (betweenness 0.43) while E and F carry none.

### 1.1 The bridge test: remove one node  *(slide 10)*

In [ ]:
for removed in ["D", "C", "E"]:
    H = G.copy(); H.remove_node(removed)
    comps = sorted((sorted(c) for c in nx.connected_components(H)), key=len, reverse=True)
    print(f"remove {removed}: {len(comps)} component(s) → {comps}")

Removing D splits the network into three pieces; removing E changes almost nothing. High-betweenness nodes can matter even when they do not have the most direct ties. On social media this is the account whose suspension, or silence, cuts one group off from another.

### 1.2 Closeness needs context in disconnected networks  *(slides 11–12)*

In [ ]:
D2 = nx.Graph([("A", "B"), ("A", "C"), ("D", "E"), ("D", "F"), ("E", "F"), ("F", "G"), ("G", "H")])
print("components:", [sorted(c) for c in nx.connected_components(D2)])
cmp = pd.DataFrame({
    "closeness (NetworkX default, corrected)": nx.closeness_centrality(D2),
    "closeness (uncorrected)": nx.closeness_centrality(D2, wf_improved=False),
}).round(3)
display(cmp)

Without the correction, A gets the **highest** closeness in the whole graph (1.0), because it is one step from both of the only two nodes it can reach. NetworkX's default (`wf_improved=True`) scales each score by how much of the graph the node can reach, so A drops. **Check components before comparing closeness** across a whole network.

### 1.3 Directed graphs: receiving vs sending, and PageRank  *(slides 7, 14–16)*

In [ ]:
DG = nx.DiGraph([("A", "D"), ("B", "D"), ("C", "D"), ("D", "E"), ("F", "E"), ("E", "G")])
directed = pd.DataFrame({"in-degree": dict(DG.in_degree()), "out-degree": dict(DG.out_degree()),
                         "PageRank (α=0.85)": nx.pagerank(DG, alpha=0.85),
                         "eigenvector": nx.eigenvector_centrality(DG, max_iter=1000)}).round(3)
display(directed.sort_values("PageRank (α=0.85)", ascending=False))

G receives only **one** link, yet has the highest PageRank: that link comes from E, which itself receives links from D (which is endorsed by three accounts). Prestige flows along the arrows. Eigenvector centrality gives a degenerate answer here: all the score drains into G (1.0) and every other node, including D, gets 0, because the graph has no cycles to send importance back. PageRank's damping (with probability 0.15 the "random surfer" jumps anywhere) handles such **dangling** nodes explicitly (slide 16), so D keeps a meaningful score.

| Metric | High score means | Typical use | Main caution |
|---|---|---|---|
| Degree | many direct ties | visible / active accounts | local only |
| Betweenness | on many shortest paths | bridge / broker accounts | assumes shortest paths matter |
| Closeness | short distance to reachable nodes | structural reachability | sensitive to disconnectedness |
| Eigenvector | connected to central nodes | prestige | connectivity assumptions |
| PageRank | receives important directed links | attention / endorsement ranking | edge direction, damping |

### 1.4 Components vs communities  *(slides 20–23)*

In [ ]:
print("connected components:", len(list(nx.connected_components(G))), "→ everyone can reach everyone")
communities = nx.community.louvain_communities(G, weight="weight", resolution=1, seed=42)
q = nx.community.modularity(G, communities, weight="weight", resolution=1)
for i, members in enumerate(communities, start=1):
    print(f"Community {i}:", sorted(members))
print("Modularity:", round(q, 3))

One component, but three communities, with modularity ≈ 0.483 (slide 29). A **component** is a hard separation (no path at all); a **community** is a soft grouping: dense inside, few ties across.

**Modularity** compares the share of edges inside communities with what a random graph with the same degrees would give. Higher = more internally connected than expected. It is useful for comparing partitions of the *same* graph, not a universal truth (slide 25).

Louvain (slide 26): every node starts alone → move nodes when it improves modularity → collapse communities into super-nodes → repeat until no gain. The **resolution** parameter changes the granularity:

In [ ]:
for res in [0.2, 0.3, 1.0, 3.0, 5.0]:
    c = nx.community.louvain_communities(G, resolution=res, seed=42)
    print(f"resolution={res}: {len(c)} communities → {[sorted(x) for x in c]}")

In [ ]:
community_id = {node: i for i, members in enumerate(communities) for node in members}
cross_edges = [(u, v) for u, v in G.edges() if community_id[u] != community_id[v]]
print("Cross-community edges (slide 38):", cross_edges)

fig, ax = plt.subplots(figsize=(5.5, 3))
nx.draw_networkx(G, POS, ax=ax, node_color=[COMM_COLORS[community_id[n]] for n in G], font_color="white",
                 node_size=[300 + 1500 * lenses.loc[n, "betweenness"] for n in G],
                 edge_color=[ORANGE if (u, v) in cross_edges or (v, u) in cross_edges else GREY for u, v in G.edges()],
                 width=[2.5 if (u, v) in cross_edges or (v, u) in cross_edges else 1 for u, v in G.edges()])
ax.set_title("Colour = community, size = betweenness, orange = cross-community edge"); ax.axis("off")
plt.tight_layout(); plt.show()

The two cross-community edges both touch D or its neighbours: structural bridges. Whether information *actually* crossed them needs temporal evidence (part 3).

### 1.5 A repost cascade  *(slides 34–40)*
A cascade is **not** the social graph: it records who retransmitted one item after whom, with time.

In [ ]:
cascade = nx.DiGraph()
cascade.add_edge("A", "B", minute=3)
cascade.add_edge("A", "C", minute=5)
cascade.add_edge("B", "D", minute=8)
cascade.add_edge("B", "E", minute=11)
cascade.add_edge("D", "F", minute=18)

def describe_cascade(tree, root, time_attr="minute"):
    """Reach, depth, breadth and speed of a propagation tree (slide 36)."""
    depth_of = nx.shortest_path_length(tree, root)                      # generation of every reached node
    times = {root: 0, **{v: d[time_attr] for _, v, d in tree.edges(data=True)}}
    per_gen = Counter(depth_of.values())
    t_sorted = sorted(t for n, t in times.items() if n != root)
    return {"reach (users)": len(depth_of) - 1, "depth (generations)": max(depth_of.values()),
            "max breadth": max(c for g, c in per_gen.items() if g > 0),
            "breadth per generation": {g: c for g, c in sorted(per_gen.items()) if g > 0},
            "time to 50% of reach": t_sorted[(len(t_sorted) - 1) // 2], "time to full reach": t_sorted[-1]}

print("Out-degree:", dict(cascade.out_degree()))
for k, v in describe_cascade(cascade, "A").items():
    print(f"{k:>22}: {v}")

In [ ]:
gen = nx.shortest_path_length(cascade, "A")
for n, g in gen.items():
    cascade.nodes[n]["gen"] = g
pos_c = nx.multipartite_layout(cascade, subset_key="gen")
fig, ax = plt.subplots(figsize=(6, 3))
nx.draw_networkx(cascade, pos_c, ax=ax, node_color=[ORANGE if n == "A" else BLUE for n in cascade], font_color="white",
                 node_size=600, edge_color=GREY, arrows=True, arrowsize=14)
nx.draw_networkx_edge_labels(cascade, pos_c, edge_labels={(u, v): f"t={d['minute']}m" for u, v, d in cascade.edges(data=True)},
                             font_size=7, ax=ax)
ax.set_title("Cascade by generation (left → right)"); ax.axis("off"); plt.tight_layout(); plt.show()

A is the seed; B is the **broadcaster** inside this cascade (out-degree 2); D continues the chain to a third generation. Without timestamps a cascade is just another static network (slide 36).

---
## 2. Do detected communities mean anything? Zachary's karate club  *(slides 22, 30)*
*"A community is a structural pattern, not automatically a demographic or opinion group. Verify with content or metadata"* (slide 22).

Zachary (1977) recorded friendships among 34 members of a university karate club. During the study the club **split in two** after a conflict between the instructor ("Mr. Hi") and the administrator ("Officer"). NetworkX ships the data, including which side each member joined. That gives us rare ground truth to check a community algorithm against.

In [ ]:
K = nx.karate_club_graph()
club = nx.get_node_attributes(K, "club")
k_comms = nx.community.louvain_communities(K, weight="weight", seed=42)
k_id = {n: i for i, c in enumerate(k_comms) for n in c}
print(f"{K.number_of_nodes()} members, {K.number_of_edges()} friendships | Louvain: {len(k_comms)} communities, "
      f"modularity {nx.community.modularity(K, k_comms, weight='weight'):.3f}")
display(pd.crosstab(pd.Series(k_id, name="Louvain community"), pd.Series(club, name="side after the split")))
print("NMI (1 = identical grouping):",
      round(normalized_mutual_info_score([club[n] for n in K], [k_id[n] for n in K]), 2))

In [ ]:
two = nx.community.louvain_communities(K, weight="weight", resolution=0.5, seed=42)
two_id = {n: i for i, c in enumerate(two) for n in c}
print(f"resolution=0.5 → {len(two)} communities, NMI = "
      f"{normalized_mutual_info_score([club[n] for n in K], [two_id[n] for n in K]):.2f}")

kb = nx.betweenness_centrality(K, weight=None)
pos_k = nx.spring_layout(K, seed=SEED)
fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx_edges(K, pos_k, ax=ax, edge_color="#d9d8d4")
nx.draw_networkx_nodes(K, pos_k, ax=ax, node_color=[COMM_COLORS[k_id[n]] for n in K],
                       node_size=[80 + 2500 * kb[n] for n in K],
                       node_shape="o", edgecolors=["black" if club[n] == "Officer" else "white" for n in K], linewidths=1.5)
nx.draw_networkx_labels(K, pos_k, ax=ax, font_size=7, font_color="white")
ax.set_title("Karate club: colour = Louvain community, size = betweenness, black ring = joined 'Officer' side")
ax.axis("off"); plt.tight_layout(); plt.show()
print("Top brokers (betweenness):", {n: round(v, 2) for n, v in sorted(kb.items(), key=lambda kv: -kv[1])[:4]})

Louvain finds **more** groups than the two real factions (it splits each side into sub-groups of friends), but almost every detected community sits on one side of the split. Lowering the resolution gives fewer, larger groups that match the split more closely. Three lessons:
- A detected community is a *structural* fact; its *meaning* (here: "Mr. Hi's side") came from metadata.
- **Resolution is a modelling choice**: report it, and choose it based on the question, not on the highest modularity.
- The biggest nodes are the instructor (0) and administrator (33), plus brokers such as 32 and 2 with friends on both sides. The one member Louvain places on the "wrong" side (member 8) has three friends on the Officer side and two on Mr. Hi's, yet joined Mr. Hi. Structure predicts most choices, not every personal decision.

---
## 3. Real data: the Higgs boson announcement on Twitter  *(slides 3–45)*
On **4 July 2012** CERN announced the discovery of a new particle consistent with the Higgs boson. Rumours had circulated since 2 July. De Domenico et al. (2013, *The Anatomy of a Scientific Rumor*) collected every tweet between 1 and 7 July 2012 containing the keywords *lhc, cern, boson* or *higgs*. SNAP publishes the interactions as one file:

`userA userB timestamp interaction` where interaction is **RT** (A retweeted B), **MT** (A mentioned B) or **RE** (A replied to B). Users are anonymised IDs; timestamps are Unix seconds (UTC).

**Stakeholder framing:** imagine CERN's communication office asking *"Who amplified our news, did it stay inside the physics bubble, and how fast did it spread?"*

In [ ]:
HIGGS_URL = "https://snap.stanford.edu/data/higgs-activity_time.txt.gz"
t0 = time.perf_counter()
events = pd.read_csv(HIGGS_URL, sep=" ", header=None, names=["source", "target", "ts", "type"])
events["time"] = pd.to_datetime(events["ts"], unit="s")
print(f"{len(events):,} events loaded in {time.perf_counter() - t0:.0f}s")
print("Interaction types:", events["type"].value_counts().to_dict())
print("Users:", pd.unique(events[["source", "target"]].values.ravel()).size)
print("Period (UTC):", events["time"].min(), "→", events["time"].max())
display(events.head())

### 3.1 Check the raw events first

In [ ]:
self_loops = events["source"] == events["target"]
print(f"Self-interactions (A → A): {self_loops.sum():,} → removed")
events = events[~self_loops].copy()
print(f"Exact duplicate events: {events.duplicated().sum():,}")

ANNOUNCEMENT = pd.Timestamp("2012-07-04 07:00")          # CERN seminar started 09:00 CEST = 07:00 UTC
hourly = events.set_index("time").groupby("type").resample("1h").size().unstack(0).fillna(0)
fig, ax = plt.subplots(figsize=(11, 3.3))
for typ, c in [("RT", BLUE), ("MT", ORANGE), ("RE", GREEN)]:
    ax.plot(hourly.index, hourly[typ], color=c, lw=1.5, label={"RT": "retweets", "MT": "mentions", "RE": "replies"}[typ])
ax.axvline(ANNOUNCEMENT, color=GREY, ls="--", lw=1); ax.text(ANNOUNCEMENT, ax.get_ylim()[1] * 0.9, " CERN seminar", fontsize=8)
ax.set_ylabel("events per hour"); ax.set_title("Activity over time (UTC)"); ax.legend()
plt.tight_layout(); plt.show()

The timeline already answers part of "how fast": activity explodes after the seminar and decays over the following days. Most activity is **retweeting**, which is why we use the retweet network for influence and diffusion.

### 3.2 Build the retweet network
Specification (week 9 habit): **node** = Twitter user; **edge** = A → B when A retweeted B; **weight** = number of retweets; **period** = 1–7 July 2012; **collection** = tweets matching the Higgs keywords (not a random sample of Twitter).

In [ ]:
rt = events[events["type"] == "RT"]
rt_edges = rt.groupby(["source", "target"]).size().rename("weight").reset_index()
RT = nx.from_pandas_edgelist(rt_edges, "source", "target", edge_attr="weight", create_using=nx.DiGraph())
print(f"Retweet network: {RT.number_of_nodes():,} users, {RT.number_of_edges():,} edges, density {nx.density(RT):.2e}")

wcc = sorted(nx.weakly_connected_components(RT), key=len, reverse=True)
print(f"Weakly connected components: {len(wcc):,}; largest = {len(wcc[0]):,} users "
      f"({len(wcc[0]) / RT.number_of_nodes():.0%}); second largest = {len(wcc[1])}")
print("Component sizes (top 10):", [len(c) for c in wcc[:10]])

**Check components first** (slide 20). Almost everyone sits in one giant weakly connected component, plus thousands of tiny ones (a user retweeting one account nobody else retweeted). Path-based measures (betweenness, closeness) only make sense *inside* a component, so we use the giant component for those.

### 3.3 Influence: five lenses, five different "top" lists
Direction matters here. In the retweet network an edge points from the **retweeter** to the **retweeted**:
- **in-degree** = how many distinct users retweeted you (*visibility*),
- **out-degree** = how many distinct users you retweeted (*activity*, not influence),
- **PageRank** = retweeted by users who are themselves retweeted (*prestige*).

In [ ]:
t0 = time.perf_counter()
infl = pd.DataFrame({
    "in-degree (retweeted by)": dict(RT.in_degree()),
    "in-strength (retweets received)": dict(RT.in_degree(weight="weight")),
    "out-degree (users retweeted)": dict(RT.out_degree()),
    "PageRank": nx.pagerank(RT, alpha=0.85, weight="weight"),
})
print(f"Computed in {time.perf_counter() - t0:.1f}s")
display(infl.sort_values("PageRank", ascending=False).head(10).style.format({"PageRank": "{:.4f}"}))
print("Top-10 by out-degree (the most active retweeters):",
      infl["out-degree (users retweeted)"].nlargest(10).index.tolist())

The heavy retweeters (top out-degree) are completely different accounts from the most retweeted ones: **activity ≠ influence** (slide 7).

Betweenness and closeness need shortest paths over the whole graph, which is expensive for 200k+ users. Two documented shortcuts:
1. compute them on an **analysis core**: users in the giant component with at least 3 distinct retweet partners (the long tail of one-off retweeters cannot be bridges anyway);
2. use **sampled betweenness** (`k` random source nodes, fixed seed): an approximation that is accurate for the top-ranked nodes.

In [ ]:
GU = RT.subgraph(wcc[0]).to_undirected()                     # undirected view for path-based measures
core_nodes = [n for n, d in GU.degree() if d >= 3]
CORE = GU.subgraph(core_nodes).copy()
CORE = CORE.subgraph(max(nx.connected_components(CORE), key=len)).copy()
print(f"Analysis core: {CORE.number_of_nodes():,} users, {CORE.number_of_edges():,} edges "
      f"({CORE.number_of_nodes() / RT.number_of_nodes():.0%} of all users)")

t0 = time.perf_counter()
bet = nx.betweenness_centrality(CORE, k=250, seed=SEED, weight=None)
print(f"Sampled betweenness (k=250) in {time.perf_counter() - t0:.0f}s")
t0 = time.perf_counter()
eig = nx.eigenvector_centrality(CORE, max_iter=1000)
print(f"Eigenvector centrality in {time.perf_counter() - t0:.0f}s")

candidates = set(infl["PageRank"].nlargest(15).index) | set(pd.Series(bet).nlargest(15).index) \
             | set(infl["in-degree (retweeted by)"].nlargest(15).index)
candidates &= set(CORE)
t0 = time.perf_counter()
clo = {n: nx.closeness_centrality(CORE, u=n) for n in candidates}       # only for the candidates: one BFS each
print(f"Closeness for {len(clo)} candidate accounts in {time.perf_counter() - t0:.0f}s")

In [ ]:
lens = pd.DataFrame({
    "in-degree": infl["in-degree (retweeted by)"],
    "PageRank": infl["PageRank"],
    "betweenness": pd.Series(bet),
    "eigenvector": pd.Series(eig),
    "closeness": pd.Series(clo),
}).loc[sorted(candidates)]
ranks = lens.rank(ascending=False).astype("Int64")
ranks.columns = [f"rank: {c}" for c in lens.columns]
table = lens.join(ranks).sort_values("rank: PageRank")
display(table.head(15).style.format({"PageRank": "{:.4f}", "betweenness": "{:.4f}", "eigenvector": "{:.3f}",
                                     "closeness": "{:.3f}"}, na_rep="–"))

In [ ]:
TOP = 20
top_sets = {m: set(lens[m].nlargest(TOP).index) for m in ["in-degree", "PageRank", "betweenness", "eigenvector"]}
overlap = pd.DataFrame({a: {b: len(top_sets[a] & top_sets[b]) for b in top_sets} for a in top_sets})
print(f"How many of the top-{TOP} accounts (among the candidates) do two lenses share?")
display(overlap)
rho = pd.DataFrame({"in-degree": infl["in-degree (retweeted by)"], "PageRank": infl["PageRank"],
                    "betweenness": pd.Series(bet), "eigenvector": pd.Series(eig)}).dropna().corr("spearman")
print("Spearman rank correlation on the analysis core:")
display(rho.round(2))

Read the two tables like slide 17 asks: *do the same nodes rank highly across metrics? If not, what structural role explains the difference?*
- **In-degree vs PageRank:** look for accounts that rank much higher on PageRank than on in-degree. They are retweeted by fewer people, but by people who are themselves retweeted a lot (prestige rather than raw visibility).
- **Betweenness:** accounts that rank high here but low on in-degree sit *between* groups without being the most retweeted: candidate brokers.
- **Eigenvector:** rewards a well-connected neighbourhood, so a less visible account inside the dense core can rank high.
- The Spearman table shows whether the lenses agree across the whole core, not just at the top. Low or negative correlations mean "important" depends heavily on the definition.

The handle names are anonymised, but in the original study the top accounts were science journalists, news outlets and physics institutions.

### 3.4 Bridges at scale: remove the top accounts

In [ ]:
def lcc_share_after_removal(graph, nodes):
    H = graph.copy(); H.remove_nodes_from(nodes)
    return len(max(nx.connected_components(H), key=len)) / graph.number_of_nodes()

rng = np.random.default_rng(SEED)
rows = []
for k in [10, 50, 100, 200]:
    rows.append({"removed": k,
                 "top betweenness": lcc_share_after_removal(CORE, pd.Series(bet).nlargest(k).index),
                 "top degree": lcc_share_after_removal(CORE, pd.Series(dict(CORE.degree())).nlargest(k).index),
                 "random users": lcc_share_after_removal(CORE, rng.choice(list(CORE), size=k, replace=False))})
robust = pd.DataFrame(rows).set_index("removed")
display(robust.style.format("{:.0%}"))

The table shows the share of the analysis core that stays in one connected piece. Compare the columns: removing random users barely changes it, while removing the same number of top-degree or top-betweenness accounts shrinks it faster. Social networks are typically robust to random loss but sensitive to the loss of hubs and bridges: slide 10 at real scale. In a retweet network, a lot of connectivity runs through a few hubs.

### 3.5 Communities in the retweet network  *(slides 24–33)*

In [ ]:
t0 = time.perf_counter()
comms = nx.community.louvain_communities(GU, weight="weight", resolution=1, seed=SEED)
comms = sorted(comms, key=len, reverse=True)
Q = nx.community.modularity(GU, comms, weight="weight")
print(f"Louvain on the giant component: {len(comms):,} communities, modularity {Q:.3f} ({time.perf_counter() - t0:.0f}s)")
sizes = pd.Series([len(c) for c in comms])
print(f"Largest 10 communities hold {sizes.head(10).sum() / sizes.sum():.0%} of users; "
      f"median community size = {sizes.median():.0f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, 31), sizes.head(30), color=BLUE)
ax.set_xlabel("community (ranked by size)"); ax.set_ylabel("users"); ax.set_title("30 largest communities")
plt.tight_layout(); plt.show()

Modularity is high, which is typical of retweet networks: people retweet a few hubs, so each hub with its audience looks like a "community". **Reproducibility check:** does the partition survive another seed or a different resolution?

In [ ]:
def as_labels(partition, nodes):
    lab = {n: i for i, c in enumerate(partition) for n in c}
    return [lab[n] for n in nodes]

nodes_order = list(GU)
base_labels = as_labels(comms, nodes_order)
rows = []
for seed, res in [(1, 1.0), (2, 1.0), (SEED, 0.5), (SEED, 2.0)]:
    t0 = time.perf_counter()
    p = nx.community.louvain_communities(GU, weight="weight", resolution=res, seed=seed)
    rows.append({"seed": seed, "resolution": res, "communities": len(p),
                 "modularity": nx.community.modularity(GU, p, weight="weight", resolution=1),
                 "NMI vs main partition": normalized_mutual_info_score(base_labels, as_labels(p, nodes_order)),
                 "seconds": time.perf_counter() - t0})
display(pd.DataFrame(rows).round(3))

A high NMI across seeds means the structure is real and not an artefact of one random run. Resolution changes the granularity (slide 26): document the value you used, and choose it for the question (a few big audiences for a briefing, many small clusters for targeting).

### 3.6 What do the communities represent? Verify with behaviour
The data has no tweet text, so we describe each large community with what we *do* have: its hub, when it was active, and how its members interacted.

In [ ]:
comm_of = {n: i for i, c in enumerate(comms) for n in c}
ev_lcc = events[events["source"].isin(comm_of)].copy()
ev_lcc["community"] = ev_lcc["source"].map(comm_of)

rows = []
for i, c in enumerate(comms[:8]):
    sub = ev_lcc[ev_lcc["community"] == i]
    hub = max(c, key=lambda n: RT.in_degree(n))
    rows.append({"community": i, "users": len(c), "hub (most retweeted)": hub,
                 "hub retweeted by": RT.in_degree(hub),
                 "share of activity before seminar": (sub["time"] < ANNOUNCEMENT).mean(),
                 "median activity time": sub["time"].median().strftime("%d %b %H:%M"),
                 "mentions+replies share": sub["type"].isin(["MT", "RE"]).mean()})
comm_profile = pd.DataFrame(rows).set_index("community")
display(comm_profile.style.format({"share of activity before seminar": "{:.0%}", "mentions+replies share": "{:.0%}"}))

How to read the profile: a community with a clearly higher share of activity **before** the seminar is a candidate for the early rumour spreaders (the physics bubble on 2–3 July); one whose activity sits almost entirely after it formed around the mainstream news. A higher share of mentions and replies suggests **conversation**, a low share suggests a pure **broadcast** audience around one hub.

This is still a *structural* label. In a real project the next step is to read the tweets of each community (slide 30, stage 2) before naming it.

### 3.7 Bridges between communities

In [ ]:
rt_lcc = rt_edges[rt_edges["source"].isin(comm_of) & rt_edges["target"].isin(comm_of)].copy()
rt_lcc["cross"] = rt_lcc["source"].map(comm_of) != rt_lcc["target"].map(comm_of)
print(f"Retweet edges crossing community boundaries: {rt_lcc['cross'].mean():.1%} "
      f"({rt_lcc.loc[rt_lcc['cross'], 'weight'].sum() / rt_lcc['weight'].sum():.1%} of retweets)")

def n_communities_touched(node):
    return len({comm_of[x] for x in nx.all_neighbors(RT, node) if x in comm_of} | {comm_of[node]})

bridge = pd.DataFrame({"betweenness": pd.Series(bet)}).nlargest(10, "betweenness")
bridge["communities touched"] = [n_communities_touched(n) for n in bridge.index]
bridge["own community"] = [comm_of[n] for n in bridge.index]
bridge["retweeted by"] = [RT.in_degree(n) for n in bridge.index]
display(bridge.style.format({"betweenness": "{:.4f}"}))

High-betweenness accounts touch many communities: they are **candidate** brokers. Whether information actually travelled *through* them is a question for the cascade data, next.

### 3.8 Diffusion: reconstructing cascades  *(slides 34–40)*
For each user we take their **first** retweet as the moment they *adopted* the news, and the account they retweeted as the **parent**. Because the whole dataset is about one topic, this traces how the *topic* travelled from user to user (the approach of the original study), not the path of one single tweet. Keeping only parent → child links where the parent adopted earlier (or is an original source) turns the retweet events into a forest of propagation trees.

In [ ]:
first_rt = (rt.sort_values("ts").drop_duplicates("source")
            .rename(columns={"source": "child", "target": "parent"})[["child", "parent", "ts"]])
adopt_time = first_rt.set_index("child")["ts"]
parent_time = first_rt["parent"].map(adopt_time)                       # NaN = parent never retweeted (original source)
valid = parent_time.isna() | (parent_time < first_rt["ts"])
tree_edges = first_rt[valid]

F = nx.DiGraph()
F.add_edges_from(zip(tree_edges["parent"], tree_edges["child"]))
nx.set_edge_attributes(F, {(p, c): t for p, c, t in tree_edges[["parent", "child", "ts"]].itertuples(index=False)}, "ts")
roots = [n for n in F if F.in_degree(n) == 0]
print(f"Adopters (users who retweeted at least once): {len(first_rt):,}; links kept: {len(tree_edges):,}; "
      f"cascade roots: {len(roots):,}")
print("Is it a forest (no cycles, one parent each)?", nx.is_forest(F))

In [ ]:
def cascade_stats(root):
    depth_of = nx.single_source_shortest_path_length(F, root)
    reached = [n for n in depth_of if n != root]
    if not reached:
        return None
    times = np.sort([F.edges[next(F.predecessors(n)), n]["ts"] for n in reached])
    start = times[0]
    per_gen = Counter(d for d in depth_of.values() if d > 0)
    return {"root": root, "reach": len(reached), "depth": max(depth_of.values()),
            "max breadth": max(per_gen.values()), "gen-1 share": per_gen[1] / len(reached),
            "start": pd.to_datetime(start, unit="s"),
            "hours to 50%": (times[(len(times) - 1) // 2] - start) / 3600,
            "hours to 90%": (times[int(0.9 * (len(times) - 1))] - start) / 3600}

t0 = time.perf_counter()
casc = pd.DataFrame([s for s in (cascade_stats(r) for r in roots) if s]).set_index("root")
print(f"Described {len(casc):,} cascades in {time.perf_counter() - t0:.0f}s")
display(casc.nlargest(10, "reach").style.format({"gen-1 share": "{:.0%}", "hours to 50%": "{:.1f}",
                                                 "hours to 90%": "{:.1f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
size_counts = casc["reach"].value_counts().sort_index()
axes[0].scatter(size_counts.index, size_counts.values, s=12, color=BLUE)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("cascade reach (log)"); axes[0].set_ylabel("number of cascades (log)"); axes[0].set_title("Cascade size distribution")
depth_counts = casc["depth"].value_counts().sort_index()
axes[1].bar(depth_counts.index.astype(str), depth_counts.values, color=ORANGE)
axes[1].set_xlabel("depth (generations)"); axes[1].set_yscale("log"); axes[1].set_title("Cascade depth")
plt.tight_layout(); plt.show()
print(f"Cascades with reach 1: {(casc['reach'] == 1).mean():.0%} | depth 1 (pure broadcast): {(casc['depth'] == 1).mean():.0%}")

Two classic findings (and one data warning):
- **Most cascades are tiny**; a handful reach thousands. Virality is rare and extremely unequal.
- **Most cascades are shallow** (depth 1–2): broadcast from a source dominates over long chains.
- ⚠️ Twitter credits a retweet to the **original author**, even if you saw it via someone else's retweet. Observed cascades are therefore *flatter* than the real path of exposure. Researchers reconstruct deeper paths with the follower graph (SNAP also publishes it). This is the "decide what the data can show" lesson from week 9 again.

### 3.9 Look at one cascade
We pick the deepest cascade among those with a reach of 30–300 users, so the picture stays readable.

In [ ]:
mid = casc[(casc["reach"].between(30, 300))]
pick = mid.sort_values(["depth", "reach"], ascending=False).index[0] if len(mid) else casc["reach"].idxmax()
T = F.subgraph(nx.descendants(F, pick) | {pick}).copy()
gens = nx.single_source_shortest_path_length(T, pick)
nx.set_node_attributes(T, gens, "gen")
stats = cascade_stats(pick)
print({k: (round(float(v), 2) if isinstance(v, (float, np.floating)) else (int(v) if isinstance(v, (int, np.integer)) else str(v)))
       for k, v in stats.items()})

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1.6, 1]})
pos_t = nx.multipartite_layout(T, subset_key="gen")
t_start = min(d["ts"] for _, _, d in T.edges(data=True))
hours = {v: (d["ts"] - t_start) / 3600 for _, v, d in T.edges(data=True)}
hours[pick] = 0
sc = nx.draw_networkx_nodes(T, pos_t, ax=axes[0], node_size=25, node_color=[hours[n] for n in T], cmap="viridis")
nx.draw_networkx_edges(T, pos_t, ax=axes[0], edge_color="#d9d8d4", arrows=False, width=0.5)
axes[0].set_title("Cascade by generation (left = seed); colour = hours after the first retweet"); axes[0].axis("off")
plt.colorbar(sc, ax=axes[0], fraction=0.03, label="hours")

reach_t = np.sort(np.array(list(hours.values()))[1:])
axes[1].step(reach_t, np.arange(1, len(reach_t) + 1), where="post", color=BLUE)
axes[1].set_xlabel("hours after first retweet"); axes[1].set_ylabel("cumulative users reached")
axes[1].set_title("Speed: reach over time")
plt.tight_layout(); plt.show()

### 3.10 Did the news cross community boundaries?
For the largest cascades: how many communities did each one reach, and how much of its reach came from **outside** the seed's own community?

In [ ]:
rows = []
for root in casc.nlargest(10, "reach").index:
    reached = nx.descendants(F, root)
    in_lcc = [n for n in reached if n in comm_of]
    home = comm_of.get(root)
    rows.append({"root": root, "reach": len(reached), "communities reached": len({comm_of[n] for n in in_lcc}),
                 "share outside seed's community": np.mean([comm_of[n] != home for n in in_lcc]) if in_lcc and home is not None else np.nan,
                 "seed betweenness": bet.get(root, np.nan)})
cross = pd.DataFrame(rows).set_index("root")
display(cross.style.format({"share outside seed's community": "{:.0%}", "seed betweenness": "{:.4f}"}, na_rep="–"))

If the biggest cascades reach many communities with a large "outside" share, the news did not stay inside one bubble. Because retweets are credited to the original source (3.8), a large "outside" share does not tell us *which* bridge carried the news across. Showing that would need exposure paths (the follower graph) and timestamps together.

### 3.11 Does centrality predict spread?  *(slides 39–41)*
*"Centrality suggests structural potential. Cascades show realized propagation."* A fair test uses **time**: measure centrality on the network **before** the seminar, then see who was retweeted most **after** it.

In [ ]:
before = rt[rt["time"] < ANNOUNCEMENT]
after = rt[rt["time"] >= ANNOUNCEMENT]
RT_before = nx.from_pandas_edgelist(before.groupby(["source", "target"]).size().rename("weight").reset_index(),
                                    "source", "target", edge_attr="weight", create_using=nx.DiGraph())
pr_before = pd.Series(nx.pagerank(RT_before, weight="weight"))
indeg_before = pd.Series(dict(RT_before.in_degree()))
received_after = after.groupby("target").size()

panel = pd.DataFrame({"PageRank before": pr_before, "in-degree before": indeg_before}).join(
    received_after.rename("retweets received after"), how="left").fillna({"retweets received after": 0})
print(f"Accounts already present before the seminar: {len(panel):,}")
print("Spearman correlation with retweets received AFTER the seminar:")
for col in ["PageRank before", "in-degree before"]:
    r, p = spearmanr(panel[col], panel["retweets received after"])
    print(f"   {col:<18} ρ = {r:.2f}")
top_before = set(panel["PageRank before"].nlargest(20).index)
top_after = set(received_after.nlargest(20).index)
print(f"Top-20 by PageRank before vs top-20 most retweeted after: {len(top_before & top_after)} accounts in common")

A positive ρ means centrality before the event is **associated** with being retweeted afterwards. The overlap of the two top-20 lists shows how imperfect that is: the accounts that are top *after* but not *before* joined (or were discovered) only when the news broke, typically official channels and big media. And even a strong correlation would not prove that position *caused* the spread (slide 41): content, timing, the platform's ranking algorithm and offline events (here, a press conference watched worldwide) all shape diffusion.

---
## 4. From numbers to a stakeholder answer  *(slides 42–45)*

| Stakeholder question | Evidence in this notebook |
|---|---|
| Which users are influential? | 3.3: defined first (visibility = in-degree, prestige = PageRank, broker = betweenness) |
| Which accounts connect separate communities? | 3.7: betweenness + communities + cross-community edges |
| Are discussions fragmented? | 3.2 components + 3.5 modularity + 3.6 behavioural profiles |
| How did a message spread? | 3.8–3.10: cascades with reach / depth / breadth / speed, plus timestamps |
| Where should communication be targeted? | community hubs (3.6) and bridges (3.7), constrained by what the data can show |

In [ ]:
largest = casc.nlargest(1, "reach").iloc[0]
shared = len(top_sets["PageRank"] & top_sets["betweenness"])
summary = f"""
CERN COMMUNICATIONS: SOCIAL MEDIA BRIEFING (1–7 July 2012, Higgs keywords)
• Scale: {len(events):,} interactions by {pd.unique(events[['source', 'target']].values.ravel()).size:,} users; \
{(events['type'] == 'RT').mean():.0%} of them retweets.
• Timing: activity peaked within hours of the seminar ({hourly.sum(axis=1).idxmax():%d %b %H:00} UTC).
• Structure: one giant connected conversation ({len(wcc[0]) / RT.number_of_nodes():.0%} of users), organised in \
{len(comms):,} communities (modularity {Q:.2f}); the 10 largest hold {sizes.head(10).sum() / sizes.sum():.0%} of users.
• Amplifiers: the top {TOP} by PageRank and by betweenness share {shared} accounts, so "most prestigious" and "best bridge" \
{"are largely the same accounts" if shared >= TOP * 0.6 else "are mostly different accounts"}.
• Spread: the largest cascade reached {int(largest['reach']):,} users in {int(largest['depth'])} generation(s); \
{(casc['reach'] == 1).mean():.0%} of cascades reached only one user.
• Caveat: retweets are credited to the original author, so cascades look shallower than real exposure paths.
"""
print(summary)

This is the shape of the slide-44 example ("Battery complaints began in Community 2, then crossed into broader discussion through two high-betweenness accounts"): **who, where, how, and what the data cannot show**.